In [1]:
from PIL import Image
import httpx
from io import BytesIO
from transformers import AutoProcessor, AutoModel
import torch

model = AutoModel.from_pretrained("google/siglip2-base-patch16-224")
processor = AutoProcessor.from_pretrained("google/siglip2-base-patch16-224")

/home/ka-lina/SigLIP2_compression/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 408/408 [00:00<00:00, 747.05it/s]


In [ ]:

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
with httpx.stream("GET", url) as response:
    image = Image.open(BytesIO(response.read()))

texts = ["a photo of 2 cats", "a photo of 2 dogs"]
# important: we pass `padding=max_length` since the model was trained with this
inputs = processor(text=texts, images=image, padding="max_length", return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

logits_per_image = outputs.logits_per_image
probs = torch.sigmoid(logits_per_image) # these are the probabilities
print(f"{probs[0][0]:.1%} that image 0 is '{texts[0]}'")

In [8]:
model

SiglipModel(
  (text_model): SiglipTextModel(
    (embeddings): SiglipTextEmbeddings(
      (token_embedding): Embedding(256000, 768)
      (position_embedding): Embedding(64, 768)
    )
    (encoder): SiglipEncoder(
      (layers): ModuleList(
        (0-11): 12 x SiglipEncoderLayer(
          (layer_norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (self_attn): SiglipAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): SiglipMLP(
            (activation_fn): GELUTanh()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768,

In [17]:
# Получаем параметры vision encoder
vision_encoder = model.vision_model
# Получаем параметры text encoder
text_encoder = model.text_model

# Функция для подсчета параметров
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

# Количество параметров в vision и text энкодерах
vision_params = count_parameters(vision_encoder)
text_params = count_parameters(text_encoder)
all_params = count_parameters(model)

# Выводим количество параметров
print(f"All params: {all_params/1_000_000:.2f} M")
print(f"Количество параметров в vision энкодере: {vision_params/1_000_000:.2f} M")
print(f"Количество параметров в text энкодере: {text_params/1_000_000:.2f} M")
print(f"{all_params/1_000_000:.2f}M (ViT: {vision_params/1_000_000:.2f}M, Text: {text_params/1_000_000:.2f}M)")

# Сравниваем
if vision_params > text_params:
    print("Vision энкодер имеет больше параметров, чем текстовый энкодер.")
else:
    print("Текстовый энкодер имеет больше параметров, чем vision энкодер.")

All params: 375.19 M
Количество параметров в vision энкодере: 92.88 M
Количество параметров в text энкодере: 282.30 M
375.19M (ViT: 92.88M, Text: 282.30M)
Текстовый энкодер имеет больше параметров, чем vision энкодер.


In [20]:
device = torch.device("cuda")
model.to(device)

torch.cuda.reset_peak_memory_stats()


url = "http://images.cocodataset.org/val2017/000000039769.jpg"
with httpx.stream("GET", url) as response:
    image = Image.open(BytesIO(response.read()))

texts = ["a photo of 2 cats", "a photo of 2 dogs"]
# important: we pass `padding=max_length` since the model was trained with this
inputs = processor(text=texts, images=image, padding="max_length", return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs.to(device))

logits_per_image = outputs.logits_per_image
probs = torch.sigmoid(logits_per_image) # these are the probabilities
print(f"{probs[0][0]:.1%} that image 0 is '{texts[0]}'")


peak_mem = torch.cuda.max_memory_allocated() / 1024**2

print(f"Peak GPU memory: {peak_mem:.2f} MB")

10.6% that image 0 is 'a photo of 2 cats'
Peak GPU memory: 1535.28 MB


In [2]:
import pandas as pd
from PIL import Image
import os
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor

In [5]:


class Flickr30kDataset(Dataset):
    def __init__(self, csv_file, image_dir, processor, split="test"):
        self.data = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.processor = processor
        self.split = split
        self.data = self.data[self.data['split'] == self.split]
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        
        # Загрузка изображения
        img_path = os.path.join(self.image_dir, row['filename'])
        image = Image.open(img_path).convert("RGB")
        
        # Получение описаний (5 описаний на одно изображение)
        captions = row['raw'][2:-2].replace('"', '').split(', ')  # Обработка списка описаний
        
        # Применение processor к изображениям и тексту
        inputs = self.processor(text=captions, images=image, return_tensors="pt", padding="max_length", max_length=64)
        # print(list(inputs.keys()))
        # print(inputs['pixel_values'].shape)
        # print(inputs['input_ids'].shape)
        all_input_ids = []
        for caption in captions:
            text_inputs = self.processor(
                text=caption, 
                images=image, 
                return_tensors="pt", 
                padding="max_length", 
                max_length=64,
                truncation=True
            )
            all_input_ids.append(text_inputs['input_ids'].squeeze(0))
        
        
        # Стекуем все 5 описаний
        input_ids = torch.stack(all_input_ids[:5])

        # print(inputs['pixel_values'][0].shape, input_ids.shape, len(captions)) 
        
        return {
            "image": inputs['pixel_values'][0],  # Только первый элемент из batch
            # "captions": captions,
            "input_ids": input_ids,  # Для текста
            # "attention_mask": inputs['attention_mask'][0]
        }

# Параметры пути
csv_file = "/mnt/d/flickr30k/flickr_annotations_30k.csv"
image_dir = "/mnt/d/flickr30k/flickr30k-images/flickr30k-images/"

# Загрузка тестовой выборки
processor = AutoProcessor.from_pretrained("google/siglip2-base-patch16-224")
dataset = Flickr30kDataset(csv_file, image_dir, processor, split="test")
dataloader = DataLoader(dataset, batch_size=128, shuffle=False)

In [112]:
print(len(dataloader))

32


In [4]:
device = torch.device("cuda")
model.to(device)


SiglipModel(
  (text_model): SiglipTextModel(
    (embeddings): SiglipTextEmbeddings(
      (token_embedding): Embedding(256000, 768)
      (position_embedding): Embedding(64, 768)
    )
    (encoder): SiglipEncoder(
      (layers): ModuleList(
        (0-11): 12 x SiglipEncoderLayer(
          (layer_norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          (self_attn): SiglipAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          (mlp): SiglipMLP(
            (activation_fn): GELUTanh()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3

In [ ]:
import torch
import torch.nn.functional as F

# Функция для вычисления cosine similarity
def cosine_similarity(a, b):
    return F.cosine_similarity(a, b, dim=1)

with torch.no_grad():
    for batch in dataloader:
        images = batch["image"].to(device)
        # print(images.shape)
        captions = batch["input_ids"].to(device)
        # print(len(captions))
        # print(captions.shape)
        image_embedding = model.get_image_features(images).pooler_output.cpu()
        text_embedding = model.get_text_features(captions).pooler_output.cpu()

        # Normalize embeddings for cosine similarity
        image_embedding = image_embedding / image_embedding.norm(p=2, dim=-1, keepdim=True)
        text_embedding = text_embedding / text_embedding.norm(p=2, dim=-1, keepdim=True)

        # print(image_embedding.shape, text_embedding.shape)

        for i in range(5):
            # print(image_embedding.shape, text_embedding[i::5].shape)
            sim = cosine_similarity(image_embedding, text_embedding[i::5])
            print(f"Cosine similarity: {sim}")

Cosine similarity: tensor([0.1723, 0.1907, 0.1819, 0.1751, 0.1606, 0.1521, 0.0983, 0.1822, 0.0407,
        0.1865, 0.1573, 0.1011, 0.1440, 0.1108, 0.1473, 0.1873, 0.1158, 0.1272,
        0.1467, 0.1432, 0.1627, 0.1883, 0.2217, 0.0948, 0.2134, 0.1576, 0.1882,
        0.1923, 0.1717, 0.1851, 0.1899, 0.1734])
Cosine similarity: tensor([0.2371, 0.1707, 0.1487, 0.0665, 0.1478, 0.1717, 0.0526, 0.1276, 0.1241,
        0.1444, 0.1731, 0.1158, 0.1320, 0.0879, 0.1621, 0.1651, 0.1122, 0.0865,
        0.1450, 0.0852, 0.1813, 0.1041, 0.1885, 0.1589, 0.2028, 0.1457, 0.1511,
        0.1928, 0.1338, 0.1835, 0.0483, 0.1668])
Cosine similarity: tensor([0.1301, 0.1727, 0.1538, 0.0752, 0.0749, 0.2009, 0.1974, 0.1009, 0.1295,
        0.1124, 0.1446, 0.0916, 0.0705, 0.1452, 0.0969, 0.1606, 0.1330, 0.1089,
        0.1361, 0.0868, 0.1786, 0.1663, 0.1805, 0.1760, 0.1888, 0.1633, 0.1222,
        0.1957, 0.0596, 0.1841, 0.0762, 0.1861])
Cosine similarity: tensor([0.1551, 0.1210, 0.1155, 0.0563, 0.0941, 0.1832, 0

KeyboardInterrupt: 

Exception ignored in: 'zmq.backend.cython._zmq.Frame.__dealloc__'
Traceback (most recent call last):
  File "zmq/backend/cython/_zmq.py", line 179, in zmq.backend.cython._zmq._check_rc
KeyboardInterrupt: 


In [10]:
text_embeddings = []
image_embeddings = []

with torch.no_grad():
    for batch in tqdm(dataloader):

        images = batch["image"].to(device)          # [B,C,H,W]
        captions = batch["input_ids"].to(device)    # [B,5,L]

        B, K, L = captions.shape

        # flatten captions
        captions = captions.view(B * K, L)

        # embeddings
        image_embed = model.get_image_features(images).pooler_output   # [B,D]
        text_embed = model.get_text_features(captions).pooler_output   # [B*5,D]

        # normalize
        image_embed = image_embed / image_embed.norm(dim=-1, keepdim=True)
        text_embed = text_embed / text_embed.norm(dim=-1, keepdim=True)

        image_embeddings.append(image_embed.cpu())
        text_embeddings.append(text_embed.cpu())

# concat
all_image_embeds = torch.cat(image_embeddings, dim=0)   # [N,D]
all_text_embeds = torch.cat(text_embeddings, dim=0)     # [N*5,D]

100%|██████████| 8/8 [01:15<00:00,  9.46s/it]


In [11]:
correct = 0
num_texts = all_text_embeds.shape[0]

# similarity matrix
sims = all_text_embeds @ all_image_embeds.T   # [N*5, N]

top1 = sims.argmax(dim=1)

for text_idx in tqdm(range(num_texts)):

    true_image_idx = text_idx // 5

    if top1[text_idx] == true_image_idx:
        correct += 1

r1_t2i = correct / num_texts

print("Recall@1 Text->Image:", r1_t2i)

100%|██████████| 5000/5000 [00:00<00:00, 135220.74it/s]

Recall@1 Text->Image: 0.6418


In [12]:
correct = 0
num_images = all_image_embeds.shape[0]

# similarity matrix
sims = all_image_embeds @ all_text_embeds.T   # [N, N*5]

top1 = sims.argmax(dim=1)

for image_idx in range(num_images):

    true_caption_indices = range(image_idx * 5, image_idx * 5 + 5)

    if top1[image_idx].item() in true_caption_indices:
        correct += 1

r1_i2t = correct / num_images

print("Recall@1 Image->Text:", r1_i2t)

Recall@1 Image->Text: 0.86
